# 1d example of embedding restriction

We show that a non-augmented model of 1d input to 1d output cannot approximate the function x^2

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader
from matplotlib.colors import to_rgb
from matplotlib.colors import LinearSegmentedColormap
from sklearn.model_selection import train_test_split



import torch
import matplotlib.pyplot as plt

# Parameters
n_samples = 300
batch_size = 10

subfolder = '1dexample'
import os
if not os.path.exists(subfolder):
    os.makedirs(subfolder)


def make_x2in1d(output_dim, n_samples = 100, plot = True, batch_size = batch_size, filename = None):
    """Generates xor
    """
    # Generate training data
    # set random seed for reproducibility
    seed = np.random.randint(1000)
    print(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    
    data = 2 * torch.rand(n_samples, 1) - 1  # shape (n_samples, 1), range [-1, 1]
    labels = data ** 2  # shape (n_samples, 1)
    # Generate outer ring points
    
    
    if plot:
                # Plot for verification
        plt.scatter(data.numpy(), labels.numpy(), alpha=0.6)
        plt.xlabel("x")
        plt.ylabel("y = x^2")
        plt.title("Sampled Data from [-1, 1] with Labels $x^2$")
        plt.grid(True)
        plt.show()
        
                # Save plot if filename provided
        if filename is not None:
            plt.savefig(f'{filename}.png', bbox_inches='tight', dpi=300)
            print(f'Plot saved as {filename}.png')
        
        plt.show()
    
    # Convert to tensors
    # data_tensor = torch.tensor(data, dtype=torch.float32)

    labels_tensor = torch.tensor(labels, dtype=torch.float32)


    data = torch.tensor(data, dtype=torch.float32)
    labels = torch.tensor(labels, dtype=torch.float32) 
    labels = torch.tensor(labels.reshape(-1, 1), dtype=torch.float32)
   
    # Create DataLoader for training data
    train_dataset = TensorDataset(data, labels)
    train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        
    return train_dataloader

train_loader = make_x2in1d(1, n_samples=n_samples)

## Plot function

In [ ]:
import plots.plots
from plots.plots import psi_manual

# ─────────────────────────────────────────────
#  Custom palette  (all colours chosen to harmonise)
# ─────────────────────────────────────────────
PALETTE = {
    # Left plot
    "model_line":    "#C0392B",   # strong red      — model curve
    "data_scatter":  "#7D9BB5",   # grey-blue        — training points

    # Middle plot
    "weight":        "#5A9E6F",   # muted green      — weights
    "bias":          "#8E9BAA",   # blue-grey         — biases

    # Right plot — eigenvalue layers cycle (no red, no grey-blue)
    "eig_layers": [
        "#7E6BC4",   # soft violet
        "#C4956A",   # sand orange
        "#4EAAAA",   # teal cyan
        "#B07BAC",   # dusty purple
        "#C4A44E",   # golden yellow
        "#5BAA8A",   # medium teal
    ],
    "zero_cross":    "#C0392B",   # red               — zero-crossing rings

    # Shared
    "grid":          "#E2E6EA",   # cool light grey grid
    "bg":            "#FFFFFF",   # pure white — axes face
    "fig_bg":        "#FFFFFF",   # pure white — figure background
    "spine":         "#434E57",   # darker cool grey-blue spines & ticks
    "text":          "#2C3E50",   # dark blue-grey text
    "zero_line_bg":  "#FFFFFF",   # white — erase grid under y=0
}


import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import torch
import torch.nn as nn

# ── Global style tweaks ──────────────────────────────────────────────────────
mpl.rcParams.update({
    "font.family":       "serif",
    "font.serif":        ["Palatino Linotype", "Palatino", "Book Antiqua", "Georgia", "DejaVu Serif"],
    "axes.facecolor":    PALETTE["bg"],
    "figure.facecolor":  PALETTE["fig_bg"],
    "axes.edgecolor":    PALETTE["spine"],
    "axes.labelcolor":   PALETTE["spine"],
    "xtick.color":       PALETTE["spine"],
    "ytick.color":       PALETTE["spine"],
    "text.color":        PALETTE["text"],
    "axes.titleweight":  "bold",
    "axes.titlesize":    14.5,
    "axes.labelsize":    12,
    "legend.fontsize":   12,
    "legend.framealpha": 0.85,
    "legend.edgecolor":  PALETTE["spine"],
    "legend.facecolor":  "#FFFFFF",
})

def _style_ax(ax):
    """Apply shared spine / grid style to an axes."""
    ax.grid(True, color=PALETTE["grid"], linewidth=0.7, zorder=0)
    for spine in ax.spines.values():
        spine.set_edgecolor(PALETTE["spine"])
        spine.set_linewidth(0.8)
    ax.tick_params(colors=PALETTE["spine"], width=0.8, length=4)


# ─────────────────────────────────────────────────────────────────────────────
def plot_params(model, ax=None, y_lim=5):
    """Plots the weights and biases of the linear layers in a model."""

    linear_layers = [m for m in model.modules() if isinstance(m, nn.Linear)]
    n_layers = len(linear_layers)

    weights, biases = [], []
    for layer in linear_layers:
        weights.append(layer.weight.detach().cpu().numpy().squeeze())
        biases.append(layer.bias.detach().cpu().numpy().squeeze())

    if ax is None:
        fig, ax = plt.subplots(figsize=(1, 3))

    if n_layers == 0:
        ax.set_title("No Linear layers found")
        return

    _style_ax(ax)

    layer_index = [i + 1 for i in range(n_layers)]

    ax.scatter(layer_index, biases,
               label='Bias', zorder=3,
               marker='^', color=PALETTE["bias"],
               edgecolors=PALETTE["text"], linewidths=0.5,
               s=60, alpha=0.88)
    ax.scatter(layer_index, weights,
               label='Weight', zorder=3,
               marker='s', color=PALETTE["weight"],
               edgecolors=PALETTE["text"], linewidths=0.5,
               s=60, alpha=0.88)

    ax.set_xlabel('Layer Index')
    ax.set_ylabel('Value')
    ax.set_title('Model Parameters', pad=10)
    ax.set_ylim(-y_lim, y_lim)
    ax.set_xlim(0.5, n_layers + 0.5)
    ax.set_xticks(layer_index)

    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12),
              ncol=1, bbox_transform=ax.transAxes)

    if ax is None:
        plt.tight_layout()
        plt.show()


# ─────────────────────────────────────────────────────────────────────────────
def plot_eigvals_new(model, ax=None, x_lim = 1):
    model.eval()
    target_ax = ax if ax is not None else plt.gca()

    eig_colors = PALETTE["eig_layers"]
    n_colors = len(eig_colors)

    for layer in range(model.num_hidden + 1):
        func = lambda x, _l=layer: model.sub_model_new(x, from_layer=_l, to_layer=_l)
        interval = torch.linspace(-x_lim, x_lim, 100)
        psi_values = np.zeros(100)

        for i, value in enumerate(interval):
            value = value.unsqueeze(0).unsqueeze(1)
            psi_values[i] = psi_manual(value, func, output_type='eigvals_1d')

        color = eig_colors[layer % n_colors]
        target_ax.plot(interval.numpy(), psi_values,
                       label=f'Layer {layer + 1}',
                       color=color, linewidth=1.8, zorder=4)

        # Zero crossings
        for i in range(len(psi_values) - 1):
            if psi_values[i] * psi_values[i + 1] <= 0:
                if psi_values[i] == 0:
                    x_zero = interval[i].item()
                elif psi_values[i + 1] == 0:
                    continue
                else:
                    x0, x1 = interval[i].item(), interval[i + 1].item()
                    y0, y1 = psi_values[i], psi_values[i + 1]
                    x_zero = x0 - y0 * (x1 - x0) / (y1 - y0)

                target_ax.plot(x_zero, 0,
                               marker='o', markerfacecolor='none',
                               markeredgecolor=PALETTE["zero_cross"],
                               markersize=9, markeredgewidth=1.8,
                               alpha=0.8, zorder=5)

    # Prominent y = 0 line
    target_ax.axhline(0, color=PALETTE["zero_line_bg"], linewidth=1.8, zorder=2.5)
    target_ax.axhline(0, color=PALETTE["text"],         linewidth=1.2,
                      linestyle='--', zorder=3, alpha=0.6)

    _style_ax(target_ax)
    target_ax.set_title('Layer Derivative', pad=10)
    target_ax.set_xlabel('Layer Input')
    target_ax.set_xlim(-x_lim, x_lim)
    target_ax.set_xticks([-1, -0.5, 0, 0.5, 1])

    if ax is None:
        target_ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
    else:
        target_ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12),
                         ncol=1, bbox_transform=target_ax.transAxes)
        target_ax.set_box_aspect(1)


# ─────────────────────────────────────────────────────────────────────────────
def subplots_model(model, epoch=None, filename=None, y_lim_params=5, subplot_margin=-0.22):
    x_plot = torch.linspace(-1, 1, 100).unsqueeze(1)
    y_plot = model(x_plot).detach().numpy()

    fig, axes = plt.subplots(
        1, 3, figsize=(15, 6),
        gridspec_kw={'width_ratios': [7, 1, 7]},
        facecolor='white'
    )

    # ── Left: model prediction + training data ────────────────────────────
    ax = axes[0]
    ax.plot(x_plot.numpy(), y_plot,
            label='ResNet', color=PALETTE["model_line"],
            linewidth=2, zorder=3)
    ax.scatter(train_loader.dataset.tensors[0].numpy(),
               train_loader.dataset.tensors[1].numpy(),
               label='Training Data', color=PALETTE["data_scatter"],
               alpha=0.75, s=24, zorder=2,
               edgecolors='none')
    ax.set_xlabel('Input')
    ax.set_ylabel('Output')
    title = (f'Approx. with ε={model.skip_param},  δ={model.residual_param},  epoch {epoch}'
             if epoch is not None else
             f'Approx. with ε={model.skip_param},  δ={model.residual_param}')
    ax.set_title(title, pad=10)
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12),
              ncol=1, bbox_transform=ax.transAxes)
    _style_ax(ax)
    ax.set_box_aspect(1)

    # ── Middle: weights & biases ──────────────────────────────────────────
    plot_params(model, ax=axes[1], y_lim=y_lim_params)

    # ── Right: eigenvalues ────────────────────────────────────────────────
    plot_eigvals_new(model, ax=axes[2])

    plt.subplots_adjust(
        left=0.05, right=0.9,
        wspace=subplot_margin,
        bottom=0.3, top=0.9
    )

    if filename is not None:
        plt.savefig(f'{filename}.png', bbox_inches='tight',
                    facecolor='white', dpi=300)
        print(f'Plot saved as {filename}.png')


# ─────────────────────────────────────────────────────────────────────────────
from models.training import train_model
import io
import imageio


def generate_gif(model, gif_name='last.gif', epochs_per_frame=10,
                 num_frames=30, fps=5, loop='yes', lr=0.01):
    frames = []

    for i in range(num_frames):
        model, acc, lossess = train_model(
            model, train_loader, train_loader,
            load_file=None, epochs=epochs_per_frame,
            lr=lr, early_stopping=False,
            cross_entropy=False, seed=None
        )

        subplots_model(model, epoch=i * epochs_per_frame)

        fig = plt.gcf()
        buf = io.BytesIO()
        fig.savefig(buf, format="png", dpi=300, facecolor='white')
        buf.seek(0)
        frames.append(imageio.imread(buf))
        plt.close(fig)

    loop_num = 0 if loop == 'no' else 1
    imageio.mimsave(gif_name, frames, fps=fps,
                    loop=loop_num, subrectangles=False)
    print(f"Saved → {gif_name}")

# One layer experiments

In [ ]:
# to reload models.resnet module after changes without restarting the kernel
import importlib
import models.resnets
import models.training
importlib.reload(models.resnets) # Reload the module
importlib.reload(models.training) # Reload the module
from models.resnets import ResNet
from models.training import compute_accuracy, train_model, train_until_threshold, plot_loss_curve

# Model Params
input_dim = 1
hidden_dim = 1
num_hidden = 1 # number of hidden layers. The total network has additionl 2 layers: input to hidden and hidden to output
output_dim = 1
activation = 'tanh' #'relu' and 'tanh' are supported (currently only weights inside the hidden layers are supported)
input_layer = False #as simple as possible, no input layer
final_sigmoid = False # True supported with binary classification only

# Training Params
load_file = None
cross_entropy = False #True
batchnorm = False
num_epochs = 100


In [ ]:
import PIL.Image
from IPython.display import Image, display

#the subplot rearrangment was hard so cropping was easier

def crop_right_side(filepath, crop_pixels):
    """
    Crops a specified number of pixels off the right side of an image 
    and overwrites the original file.
    """
    filepath = filepath + '.png'
    # Open the saved plot
    img = PIL.Image.open(filepath)
    width, height = img.size
    
    # The crop box is a tuple of (left, upper, right, lower)
    # We keep left (0), upper (0), and lower (height), but pull in the right edge
    new_right = width - crop_pixels
    crop_box = (0, 0, new_right, height)
    
    # Crop and save back to the same path
    cropped_img = img.crop(crop_box)
    cropped_img.save(filepath)
    # print(f"Cropped {crop_pixels}px from the right of {filepath}")

In [ ]:
mlp = ResNet(input_dim=input_dim, hidden_dim=hidden_dim, output_dim=output_dim, num_hidden=num_hidden, activation=activation, skip_param=0, final_sigmoid=final_sigmoid, input_layer=input_layer, batchnorm=batchnorm)

filename = subfolder + '/mlp'

mlp, acc_base, losses_base = train_model(mlp,
    train_loader, train_loader, load_file = filename, epochs=num_epochs, lr=0.01, early_stopping = False, cross_entropy=False, seed = None, save_path = filename)

plot_loss_curve(losses_base, title=f"Base Model Loss Curve", filename = 'ff1d')

subplots_model(mlp, epoch = None, filename= filename, y_lim_params=5, subplot_margin=-0.15)

crop_right_side(filename, crop_pixels=1250)

display(Image(filename= filename + '.png', width=500))



In [ ]:
resnet_skip1 = ResNet(input_dim=input_dim, hidden_dim=hidden_dim, output_dim=output_dim, num_hidden=num_hidden, activation=activation, skip_param=1, final_sigmoid=final_sigmoid, input_layer=input_layer, batchnorm=batchnorm)

num_epochs = 300

filename= subfolder + '/resnet_skip1'

resnet_skip1, acc_skip1, losses_skip1 = train_model(resnet_skip1,
    train_loader, train_loader, load_file = None, epochs=num_epochs, lr=0.01, early_stopping = False, cross_entropy=False, seed = None)

plot_loss_curve(losses_base, title=f"Base Model Loss Curve", filename = 'ff1d')


subplots_model(resnet_skip1, epoch = None, filename= filename, y_lim_params=5, subplot_margin=-0.15)

crop_right_side(filename, crop_pixels=1250)

display(Image(filename= filename + '.png', width=500))


In [ ]:
import io
import imageio      # v2 API keeps .mimsave unchanged
import matplotlib.pyplot as plt
import torch
 
# -----------------------------  CONFIG  ---------------------------------
gif_name = subfolder + "/skip0_training.gif"
fps = 1                    # 5 frames ≈ 200 ms between frames
frames = []                # collects RGB arrays for GIF
# ------------------------------------------------------------------------

hidden_dim   = 1
num_hidden   = 1
skip_param   = 0
num_frames = 20

model_skip0 = ResNet(input_dim=input_dim,
                     hidden_dim=hidden_dim,
                     output_dim=output_dim,
                     num_hidden=num_hidden,
                     activation=activation,
                     skip_param=skip_param, final_sigmoid=final_sigmoid, input_layer=input_layer)



# generate_gif(model_skip0, gif_name = gif_name, num_frames = 1)
# subplots_model(model_skip0)

In [ ]:
from IPython.display import Image, display

# display(Image(filename=subfolder + "/skip0_training.gif", width=500))

In [ ]:
import io
import imageio      # v2 API keeps .mimsave unchanged
import matplotlib.pyplot as plt
import torch

# -----------------------------  CONFIG  ---------------------------------
gif_name = subfolder + "/skip1_training.gif"
fps = 5                    # 5 frames ≈ 200 ms between frames
frames = []                # collects RGB arrays for GIF
# ------------------------------------------------------------------------

hidden_dim   = 1
num_hidden   = 1
skip_param   = 1

model_skip1 = ResNet(input_dim=input_dim,
                     hidden_dim=hidden_dim,
                     output_dim=output_dim,
                     num_hidden=num_hidden,
                     activation=activation,
                     skip_param=skip_param, final_sigmoid=final_sigmoid, input_layer=input_layer)

# generate_gif(model_skip1, gif_name=gif_name)

In [ ]:
# display(Image(filename=subfolder + "/skip1_training.gif", width=600))

# Two Layers

In [ ]:
import models.resnets; importlib.reload(models.resnets)
from models.resnets import ResNet


In [ ]:
import models.resnets; importlib.reload(models.resnets)
from models.resnets import ResNet

skip_param = 1
residual_param = 1
hidden_dim = 1
num_hidden = 2

model = ResNet(input_dim=input_dim,
                     hidden_dim=hidden_dim,
                     output_dim=output_dim,
                     num_hidden=num_hidden,
                     activation=activation,
                     skip_param=skip_param, final_sigmoid=final_sigmoid, batchnorm=False, input_layer=input_layer)

num_epochs = 200

model, acc, lossess = train_model(model,
    train_loader, train_loader, load_file = None, epochs=num_epochs, lr=0.01, early_stopping = False, cross_entropy=False, seed = None)

subplots_model(model, epoch = None, filename= subfolder + '/2l_eps1delta1')
# generate_gif(model, gif_name = subfolder + '/1d2l_eps1delta1.gif', num_frames = 50)
# display(Image(filename=subfolder + '/1d2l_eps1delta1.gif', width=900))
    

In [ ]:
model = ResNet(input_dim=input_dim,
                     hidden_dim=hidden_dim,
                     output_dim=output_dim,
                     num_hidden=num_hidden,
                     activation=activation,
                     skip_param=skip_param, final_sigmoid=final_sigmoid, batchnorm=False, input_layer=input_layer)

model, acc, losses = train_model(model,
    train_loader, train_loader, load_file = subfolder + '/1d2l_e1d1_best', epochs=num_epochs, lr=0.01, early_stopping = False, cross_entropy=False, seed = None)

model, acc, losses = train_model(model,
    train_loader, train_loader, load_file = None, epochs=num_epochs, lr=0.01, early_stopping = False, cross_entropy=False, seed = None)


subplots_model(model, epoch = None, filename= subfolder + '/2l_eps1delta1_best')

In [ ]:
import models.resnets; importlib.reload(models.resnets)
from models.resnets import ResNet

skip_param = 20
residual_param = 20
hidden_dim = 1
num_hidden = 2

filename = subfolder + '/2l_eps30delta30'

model = ResNet(input_dim=input_dim,
                     hidden_dim=hidden_dim,
                     output_dim=output_dim,
                     num_hidden=num_hidden,
                     activation=activation,
                     skip_param=skip_param,
                     residual_param = residual_param,
                     final_sigmoid=final_sigmoid, batchnorm=False, input_layer=input_layer)

num_epochs = 200

model, acc, lossess = train_model(model,
    train_loader, train_loader, load_file = filename, epochs=num_epochs, lr=0.01, early_stopping = False, cross_entropy=False, seed = None, save_path = filename)


plot_loss_curve(lossess, title=f"Loss Curve for {filename}", filename = filename + '_loss_curve')
subplots_model(model, epoch = None, filename = filename, y_lim_params=8)
# generate_gif(model, gif_name = subfolder + '/1d2l_eps1delta1.gif', num_frames = 50)
# display(Image(filename=subfolder + '/1d2l_eps1delta1.gif', width=900))

    

In [ ]:
import models.resnets; importlib.reload(models.resnets)
from models.resnets import ResNet

skip_param = 1
residual_param = 0.3
hidden_dim = 1
num_hidden = 2

filename = subfolder + '/2l_eps1delta0p3'

model = ResNet(input_dim=input_dim,
                     hidden_dim=hidden_dim,
                     output_dim=output_dim,
                     num_hidden=num_hidden,
                     activation=activation,
                     skip_param=skip_param,
                     residual_param = residual_param,
                     final_sigmoid=final_sigmoid, batchnorm=False, input_layer=input_layer)


with torch.no_grad():
    # First Hidden Layer
    model.res_blocks[0].fc.weight.copy_(torch.tensor([[-5.]]))
        
#     # Second Hidden Layer
    model.res_blocks[1].fc.weight.copy_(torch.tensor([[5.]]))
    
    # --- Output Layer ---
    # Shape of weight: [output_dim, hidden_dim] -> [1, 1]
    
num_epochs = 300

model, acc, lossess = train_model(model,
    train_loader, train_loader, load_file = filename, epochs=num_epochs, lr=0.01, early_stopping = False, cross_entropy=False, seed = None, save_path = filename)

subplots_model(model, epoch = None, filename = filename, y_lim_params=19)
# generate_gif(model, gif_name = subfolder + '/1d2l_eps1delta1.gif', num_frames = 50)
# display(Image(filename=subfolder + '/1d2l_eps1delta1.gif', width=900))

    

In [ ]:
import models.resnets; importlib.reload(models.resnets)
from models.resnets import ResNet

skip_param = 2
residual_param = 1
hidden_dim = 1
num_hidden = 2

filename = subfolder + '/2l_eps2delta1'

model = ResNet(input_dim=input_dim,
                     hidden_dim=hidden_dim,
                     output_dim=output_dim,
                     num_hidden=num_hidden,
                     activation=activation,
                     skip_param=skip_param,
                     residual_param = residual_param,
                     final_sigmoid=final_sigmoid, batchnorm=False, input_layer=input_layer)

num_epochs = 200

model, acc, lossess = train_model(model,
    train_loader, train_loader, load_file = filename, epochs=num_epochs, lr=0.01, early_stopping = False, cross_entropy=False, seed = None, save_path = filename)

subplots_model(model, epoch = None, filename = filename, y_lim_params=8)
# generate_gif(model, gif_name = subfolder + '/1d2l_eps1delta1.gif', num_frames = 50)
# display(Image(filename=subfolder + '/1d2l_eps1delta1.gif', width=900))
    

In [ ]:
skip_param = 1
residual_param = 10
hidden_dim = 1
num_hidden = 2

filename = subfolder + '/2l_eps1delta10'

model = ResNet(input_dim=input_dim,
                     hidden_dim=hidden_dim,
                     output_dim=output_dim,
                     num_hidden=num_hidden,
                     activation=activation,
                     skip_param=skip_param,
                     residual_param = residual_param,
                     final_sigmoid=final_sigmoid, batchnorm=False, input_layer=input_layer)

num_epochs = 300

model, acc, lossess = train_model(model,
    train_loader, train_loader, load_file = filename, epochs=num_epochs, lr=0.01, early_stopping = False, cross_entropy=False, seed = None, save_path = filename)

subplots_model(model, epoch = None, filename = filename)
# generate_gif(model, gif_name = subfolder + '/1d2l_eps1delta1.gif', num_frames = 50)
# display(Image(filename=subfolder + '/1d2l_eps1delta1.gif', width=900))
    

In [ ]:
skip_param = 0
residual_param = 1
hidden_dim = 1
num_hidden = 2

filename = subfolder + '/2l_eps0delta1'

model = ResNet(input_dim=input_dim,
                     hidden_dim=hidden_dim,
                     output_dim=output_dim,
                     num_hidden=num_hidden,
                     activation=activation,
                     skip_param=skip_param,
                     residual_param = residual_param,
                     final_sigmoid=final_sigmoid, batchnorm=False, input_layer=input_layer)

num_epochs = 300

model, acc, lossess = train_model(model,
    train_loader, train_loader, load_file = filename, epochs=num_epochs, lr=0.01, early_stopping = False, cross_entropy=False, seed = None, save_path = filename)

subplots_model(model, epoch = None, filename = filename)
# generate_gif(model, gif_name = subfolder + '/1d2l_eps1delta1.gif', num_frames = 50)
# display(Image(filename=subfolder + '/1d2l_eps1delta1.gif', width=900))
    

In [ ]:
skip_param = 0.01
residual_param = 1
hidden_dim = 1
num_hidden = 2

filename = subfolder + '/2l_eps0p1delta1'

model = ResNet(input_dim=input_dim,
                     hidden_dim=hidden_dim,
                     output_dim=output_dim,
                     num_hidden=num_hidden,
                     activation=activation,
                     skip_param=skip_param,
                     residual_param = residual_param,
                     final_sigmoid=final_sigmoid, batchnorm=False, input_layer=input_layer)

num_epochs = 300

model, acc, lossess = train_model(model,
    train_loader, train_loader, load_file = filename, epochs=num_epochs, lr=0.01, early_stopping = False, cross_entropy=False, seed = None, save_path = filename)

subplots_model(model, epoch = None, filename = filename, y_lim_params= 8)
# generate_gif(model, gif_name = subfolder + '/1d2l_eps1delta1.gif', num_frames = 50)
# display(Image(filename=subfolder + '/1d2l_eps1delta1.gif', width=900))
    

In [ ]:
skip_param = 0.01
residual_param = 1
hidden_dim = 1
num_hidden = 2

filename = subfolder + '/2l_eps0p1delta1b'

model = ResNet(input_dim=input_dim,
                     hidden_dim=hidden_dim,
                     output_dim=output_dim,
                     num_hidden=num_hidden,
                     activation=activation,
                     skip_param=skip_param,
                     residual_param = residual_param,
                     final_sigmoid=final_sigmoid, batchnorm=False, input_layer=input_layer)

num_epochs = 300

# with torch.no_grad():
#     # First Hidden Layer
#     model.res_blocks[0].fc.weight.copy_(torch.tensor([[-1.]]))

# model, acc, lossess = train_model(model,
#     train_loader, train_loader, load_file = filename, epochs=num_epochs, lr=0.01, early_stopping = False, cross_entropy=False, seed = None, save_path = filename)

model, acc, lossess = train_model(model,
    train_loader, train_loader, load_file = filename, epochs=num_epochs, lr=0.01, early_stopping = False, cross_entropy=False, seed = None, save_path = filename)

subplots_model(model, epoch = None, filename = filename, y_lim_params= 8)
# generate_gif(model, gif_name = subfolder + '/1d2l_eps1delta1.gif', num_frames = 50)
# display(Image(filename=subfolder + '/1d2l_eps1delta1.gif', width=900))
    

In [ ]:
import models.resnets; importlib.reload(models.resnets)
from models.resnets import ResNet

counter = 0

for skip in [1, 1, 1]:
    for sara in [1, 1, 1]:
        skip_param = skip
        residual_param = sara
        hidden_dim = 1
        num_hidden = 2

        model_running = ResNet(input_dim=input_dim,
                            hidden_dim=hidden_dim,
                            output_dim=output_dim,
                            num_hidden=num_hidden,
                            activation=activation,
                            skip_param=skip_param, residual_param=residual_param, final_sigmoid=final_sigmoid, batchnorm=False, input_layer=input_layer)
        gif_name = subfolder + '/running' + str(counter) + '.gif'
        generate_gif(model_running, gif_name = gif_name, fps = 2, num_frames = 50)
        counter += 1
        display(Image(filename=gif_name, width=900))
    

In [ ]:
import models.resnets; importlib.reload(models.resnets)
from models.resnets import ResNet

counter = 0

for skip in [1, 1]:
    for sara in [1, 1]:
        skip_param = skip
        residual_param = sara
        hidden_dim = 1
        num_hidden = 3

        model_running = ResNet(input_dim=input_dim,
                            hidden_dim=hidden_dim,
                            output_dim=output_dim,
                            num_hidden=num_hidden,
                            activation=activation,
                            skip_param=skip_param, residual_param=residual_param, final_sigmoid=final_sigmoid, batchnorm=False, input_layer=input_layer)
        gif_name = subfolder + '/running' + str(counter) + '.gif'
        generate_gif(model_running, gif_name = gif_name, fps = 2, num_frames = 50)
        counter += 1
        display(Image(filename=gif_name, width=900))
    

In [ ]:
import models.resnets; importlib.reload(models.resnets)
from models.resnets import ResNet, ResidualBlock

counter = 0

for skip in [1]:
    for sara in [0.1]:
        skip_param = skip
        residual_param = sara
        hidden_dim = 1
        num_hidden = 2

        model_running = ResNet(input_dim=input_dim,
                            hidden_dim=hidden_dim,
                            output_dim=output_dim,
                            num_hidden=num_hidden,
                            activation=activation,
                            skip_param=skip_param, residual_param=residual_param, final_sigmoid=final_sigmoid, batchnorm=False, input_layer=input_layer)
        
        # for reproducibility (optional)

# find all ResidualBlocks inside the model
        blocks = [m for m in model_running.modules() if isinstance(m, ResidualBlock)]


        with torch.no_grad():
            # for block in blocks:
            #     nn.init.xavier_normal_(block.fc.weight, gain=1/residual_param)
        

            blocks[0].fc.weight[0] = -2/residual_param
            blocks[1].fc.weight[0] = -0.5/residual_param
    
            
        gif_name = subfolder + '/running' + str(counter) + '.gif'
        
        
        
        generate_gif(model_running, gif_name = gif_name, fps = 2, epochs_per_frame = 1, num_frames = 40, lr = 0.01)
        counter += 1
        display(Image(filename=gif_name, width=900))
    